# Badge 1: Data Warehousing Workshop: Lesson 9: Staging Data

MY_INTERNAL_STAGE, file formats, COPY INTO, and the two challenge loads.

**How to read this notebook**
- Each `## Page:` heading is one page of the Rise lesson, in course order.
- `> Course:` cells quote what the learner is told to do (text or screenshot steps).
- Code cells run that task. Course code is verbatim except role/warehouse names, which are swapped for `BADGE1_DWW_ADMIN` (ACCOUNTADMIN), `BADGE1_DWW_ROLE` (SYSADMIN) and `BADGE1_DWW_WH` (COMPUTE_WH).
- Cells marked **SQL equivalent** replace a UI-only step; **Derived solution** marks challenge-lab answers the course does not give.
- DORA checks are not run here; see `dora/dora_tests.ipynb`.
- Files the course has learners download are on the `@UTIL_DB.PUBLIC.BADGE1_DOWNLOADS` stage, which stands in for the learner's Downloads folder.

Prerequisite: run `dora/dora_setup.ipynb` once.

In [ ]:
# Isolation mapping (set once per notebook; SQL cells use these via {{...}})
ADMIN_ROLE    = "BADGE1_DWW_ADMIN"   # stands in for the course's ACCOUNTADMIN
SYSADMIN_ROLE = "BADGE1_DWW_ROLE"    # stands in for the course's SYSADMIN
WAREHOUSE     = "BADGE1_DWW_WH"      # stands in for the course's COMPUTE_WH

In [ ]:
use role {{ADMIN_ROLE}};
use secondary roles none;
use warehouse {{WAREHOUSE}};
select current_role(), current_warehouse();

## Page: 🥋 Create a Snowflake Stage Object

> **Course:** UI steps (DWW_145–147): UTIL_DB / PUBLIC → **Create** → **Stage** → **Snowflake Managed** → Stage Name **my_internal_stage**, schema **UTIL_DB.PUBLIC**, Comment **a place to hold files before loading them**, Directory table **on**, Encryption **Server-side** → **Create**. The dialog says *Creating as ACCOUNTADMIN*.

In [ ]:
-- SQL equivalent of the Create Stage dialog
use role {{ADMIN_ROLE}};
create stage util_db.public.my_internal_stage
  comment = 'a place to hold files before loading them'
  directory = (enable = true)
  encryption = (type = 'SNOWFLAKE_SSE');

### 🤖 DORA check DWW10 goes here
The course has the learner run **DWW10** (Internal stage created) at this point.
It is in `dora/dora_tests.ipynb`, section **DWW10**, as a local check plus the exact DORA code.

## Page: 🥋 Load a File into Your Stage

> **Course:** Download a file (VEG_NAME_TO_SOIL_TYPE_PIPE.txt) & upload it to your stage (DWW_149, DWW_150: stage → + Files → browse → Upload).

In [ ]:
-- SQL equivalent of uploading the downloaded file(s) through the stage's + Files dialog
use role {{ADMIN_ROLE}};
copy files into @util_db.public.my_internal_stage
from @UTIL_DB.PUBLIC.BADGE1_DOWNLOADS
files = ('VEG_NAME_TO_SOIL_TYPE_PIPE.txt');
list @util_db.public.my_internal_stage;

The course's COPY INTO template (illustrative, not run):

```sql
copy into my_table_name
from @my_internal_stage
files = ( 'IF_I_HAD_A_FILE_LIKE_THIS.txt')
file_format = ( format_name='EXAMPLE_FILEFORMAT' );
```

## Page: 🥋 Use the COPY INTO Statement to Load Data

> **Course:** Create a Table for Soil Types. Make sure you create it in the GARDEN_PLANTS database, in the VEGGIES schema. (Role not specified.)

> ⚠️ **Audit note:** The stage was created as ACCOUNTADMIN, so SYSADMIN cannot read it. Where the course does not name a role, these cells use the ACCOUNTADMIN stand-in so the COPY INTO succeeds as it would for a trial user on the default role.

In [ ]:
use role {{ADMIN_ROLE}};
use schema GARDEN_PLANTS.VEGGIES;
create or replace table vegetable_details_soil_type
( plant_name varchar(25)
 ,soil_type number(1,0)
);

In [ ]:
create file format garden_plants.veggies.PIPECOLSEP_ONEHEADROW 
    type = 'CSV'--csv is used for any flat file (tsv, pipe-separated, etc)
    field_delimiter = '|' --pipes as column separators
    skip_header = 1 --one header row to skip
    ;

In [ ]:
copy into vegetable_details_soil_type
from @util_db.public.my_internal_stage
files = ( 'VEG_NAME_TO_SOIL_TYPE_PIPE.txt')
file_format = ( format_name=GARDEN_PLANTS.VEGGIES.PIPECOLSEP_ONEHEADROW );

### 🤖 DORA check DWW11 goes here
The course has the learner run **DWW11** (Veg Det Soil Type Count, expects 42) at this point.
It is in `dora/dora_tests.ipynb`, section **DWW11**, as a local check plus the exact DORA code.

## Page: 🥋 Query a File Before You Load It

> **Course:** 🎯 Download and upload this file (LU_SOIL_TYPE.tsv) into your stage.

In [ ]:
-- SQL equivalent of uploading the downloaded file(s) through the stage's + Files dialog
use role {{ADMIN_ROLE}};
copy files into @util_db.public.my_internal_stage
from @UTIL_DB.PUBLIC.BADGE1_DOWNLOADS
files = ('LU_SOIL_TYPE.tsv');
list @util_db.public.my_internal_stage;

In [ ]:
create file format garden_plants.veggies.COMMASEP_DBLQUOT_ONEHEADROW 
    TYPE = 'CSV'--csv for comma separated files
    FIELD_DELIMITER = ',' --commas as column separators
    SKIP_HEADER = 1 --one header row  
    FIELD_OPTIONALLY_ENCLOSED_BY = '"'
--this means that some values will be wrapped in double-quotes bc they have commas in them
    ;

> **Course:** Explore the Effect of File Formats On Data Interpretation.

In [ ]:
--The data in the file, with no FILE FORMAT specified
select $1
from @util_db.public.my_internal_stage/LU_SOIL_TYPE.tsv

In [ ]:
--Same file but with one of the file formats we created earlier  
select $1, $2, $3
from @util_db.public.my_internal_stage/LU_SOIL_TYPE.tsv
(file_format => garden_plants.veggies.COMMASEP_DBLQUOT_ONEHEADROW)

In [ ]:
--Same file but with the other file format we created earlier
select $1, $2, $3
from @util_db.public.my_internal_stage/LU_SOIL_TYPE.tsv
(file_format => garden_plants.veggies.PIPECOLSEP_ONEHEADROW )

## Page: 🎯 Challenge Lab: Create a File Format, Use it in a COPY INTO

> **Course:** Create a file format that will help you load files with these properties. Name the file format: **L9_CHALLENGE_FF**. Make sure the data looks like the screenshot (DWW_154) when you use your new File Format in the query. Tip: if columns were separated by a tab, you would put FIELD_DELIMITER = '\t'.

In [ ]:
-- Derived solution: tab-delimited, one header row; TRIM_SPACE handles the padded values in the file
create or replace file format garden_plants.veggies.L9_CHALLENGE_FF
    type = 'CSV'
    field_delimiter = '\t'
    skip_header = 1
    trim_space = true;

In [ ]:
select $1, $2, $3
from @util_db.public.my_internal_stage/LU_SOIL_TYPE.tsv
(file_format => garden_plants.veggies.L9_CHALLENGE_FF);

> **Course:** Create a Soil Type Look Up Table in GARDEN_PLANTS.VEGGIES. Also, make sure it is owned by the SYSADMIN role.

In [ ]:
use role {{SYSADMIN_ROLE}};
use schema GARDEN_PLANTS.VEGGIES;
create or replace table LU_SOIL_TYPE(
SOIL_TYPE_ID number,	
SOIL_TYPE varchar(15),
SOIL_DESCRIPTION varchar(75)
);

> **Course:** 🎯 Create a COPY INTO command to load LU_SOIL_TYPE.tsv from your stage to the LU_SOIL_TYPE table. DO NOT use 'SKIP_FILE' or 'CONTINUE' for ON_ERROR. Run a SELECT * to see if it loaded nicely.

In [ ]:
-- Derived solution (run as the ACCOUNTADMIN stand-in, which can read the stage)
use role {{ADMIN_ROLE}};
copy into garden_plants.veggies.lu_soil_type
from @util_db.public.my_internal_stage
files = ('LU_SOIL_TYPE.tsv')
file_format = (format_name = garden_plants.veggies.L9_CHALLENGE_FF);

In [ ]:
select * from garden_plants.veggies.lu_soil_type;

## Page: 🎯 Challenge Lab: Create a Table, Use It in a COPY INTO!

> **Course:** Create a table called VEGETABLE_DETAILS_PLANT_HEIGHT in the VEGGIES schema, using the file's header row for column names and good data types. Upload veg_plant_height.csv to your stage. Choose an existing file format. Use a COPY INTO command to load it.

In [ ]:
-- Derived solution: header is plant_name, UOM, Low_End_of_Range, High_End_of_Range
use role {{SYSADMIN_ROLE}};
create or replace table garden_plants.veggies.VEGETABLE_DETAILS_PLANT_HEIGHT (
    plant_name varchar(25),
    UOM varchar(1),
    Low_End_of_Range number(2,0),
    High_End_of_Range number(2,0)
);

In [ ]:
-- SQL equivalent of uploading the downloaded file(s) through the stage's + Files dialog
use role {{ADMIN_ROLE}};
copy files into @util_db.public.my_internal_stage
from @UTIL_DB.PUBLIC.BADGE1_DOWNLOADS
files = ('veg_plant_height.csv');
list @util_db.public.my_internal_stage;

In [ ]:
-- Derived solution: the existing comma-separated file format fits this file
use role {{ADMIN_ROLE}};
copy into garden_plants.veggies.VEGETABLE_DETAILS_PLANT_HEIGHT
from @util_db.public.my_internal_stage
files = ('veg_plant_height.csv')
file_format = (format_name = garden_plants.veggies.COMMASEP_DBLQUOT_ONEHEADROW);

In [ ]:
select count(*) as rows_loaded from garden_plants.veggies.VEGETABLE_DETAILS_PLANT_HEIGHT;

### 🤖 DORA check DWW12 goes here
The course has the learner run **DWW12** (Veg Detail Plant Height Count, expects 41) at this point.
It is in `dora/dora_tests.ipynb`, section **DWW12**, as a local check plus the exact DORA code.

### 🤖 DORA check DWW13 goes here
The course has the learner run **DWW13** (Soil Type Look Up Table, expects 8) at this point.
It is in `dora/dora_tests.ipynb`, section **DWW13**, as a local check plus the exact DORA code.

### 🤖 DORA check DWW14 goes here
The course has the learner run **DWW14** (Challenge File Format Created) at this point.
It is in `dora/dora_tests.ipynb`, section **DWW14**, as a local check plus the exact DORA code.

## Page: 👂 Check In With DORA

> **Course:** Use the YSA app (https://ysa.snowflakeuniversity.com) to see which tests DORA has received and whether they passed and were valid. Outside Snowflake; nothing to run.